In [23]:
import numpy as np
import pandas as pd

# Lock the seed. 
np.random.seed(42)

# scale of our dataset.
NUM_HOUSES = 200
NUM_USERS = 50

#options in New Zealand.
REGIONS = ['Central Auckland', 'North Shore', 'West Auckland', 'South Auckland']
PROPERTY_TYPES = ['House', 'Townhouse', 'Apartment']
OUTSIDE_SPACES = ['None', 'Balcony', 'Deck Only', 'Small Courtyard', 'Large Backyard']

#generate house id's.
house_id = np.arange(1000, 1000 + NUM_HOUSES)

#probability of houses being in central, north, west, south.
house_regions = np.random.choice(REGIONS, size = NUM_HOUSES, p = [0.4, 0.2, 0.2, 0.2])

#property type.
#House = 50% , Apartment = 30%, Townhouse = 20%
prop_type = np.random.choice(PROPERTY_TYPES, size = NUM_HOUSES, p = [0.5, 0.3, 0.2])

#bedrooms.
bedrooms = np.random.randint(1, 6, size = NUM_HOUSES)

#school ratings.
schools = np.random.randint(1, 11, size = NUM_HOUSES)

#outside spaces
outside_spaces = []
#price
price = []

#Regional multipliers
regional_multipliers = {
    'Central Auckland' : 1.52,
    'North Shore': 1.2,
    'West Auckland': 0.9,
    'South Auckland': 0.8
}

dispatch_table = {
    'Apartment': {
        'outside_spaces' : ['None', 'Balcony'],
        'probability' : [0.4, 0.6]
    },
    'Townhouse': {
        'outside_spaces': ['Deck Only', 'Small Courtyard'],
        'probability': [0.5, 0.5]
    },
    'House': {
        'outside_spaces': ['Small Courtyard', 'Large Backyard'],
        'probability': [0.2, 0.8]
    }
}

for i in range(NUM_HOUSES):
    #current type
    current_type = prop_type[i]
    
    if current_type in dispatch_table:
        # config a dictionary
        config = dispatch_table[current_type]
        space = np.random.choice(config['outside_spaces'], p=config['probability'])
    outside_spaces.append(space)

    # calculating the price.
    base_price = 300000

    #price is 150000 for each bedroom
    base_price += (bedrooms[i]*150000)

    #premium schooling areas is 25000 for each point.
    base_price += (schools[i]*25000)

    #final price = regional multipliers * base price 
    final_price = regional_multipliers[house_regions[i]] * base_price

    #considering the market fluctuations
    market_fluctuations = np.random.uniform(0.85, 1.15)
    final_price = int(market_fluctuations * final_price)
    price.append(final_price)

price = np.array(price)
cv_variance = np.random.uniform(0.90, 1.10, size = NUM_HOUSES)
cv_values = (price * cv_variance).astype(int)


houses_df = pd.DataFrame({
    'house_id': house_id,
    'region': house_regions,
    'property_type': prop_type,
    'price': price,
    'cv': cv_values,
    'bedrooms': bedrooms,
    'school_rating': schools,
    'outside_space': outside_spaces
})

user_id = np.arange(1, 1 + NUM_USERS)

# randomly assign region preferance for each user
pref_region = np.random.choice(REGIONS, size = NUM_USERS)

#min bedrooms required by the user
pref_beds = np.random.randint(1, 5, size = NUM_USERS)


#generating max budeget based on the region
max_budget = []
prem_regions = {
    'Central Auckland': np.random.randint(900000, 3000000),
    'North Shore': np.random.randint(800000, 2200000),
    'West Auckland': np.random.randint(500000, 1500000),
    'South Auckland': np.random.randint(500000, 1500000)
}

for r in pref_region:
    if r in prem_regions:
        budget = prem_regions[r]
    max_budget.append(budget)

users_df = pd.DataFrame({
    'user_id': user_id,
    'preferred_region': pref_region,
    'max_budget': max_budget,
    'min_bedrooms': pref_beds
})


interactions = []

for _, user in users_df.iterrows():
    for _, house in houses_df.iterrows():
        
        # Calculate behavioral matches
        region_match = (user['preferred_region'] == house['region'])
        affordable = (house['price'] <= user['max_budget'])
        space_match = (house['bedrooms'] >= user['min_bedrooms'])
        
        # Scenario A: Perfect Match (Fits region, budget, and size)
        if region_match and affordable and space_match:
            # 85% chance they watchlist it, 15% chance they miss it or dislike style
            label = np.random.choice([1, 0], p=[0.85, 0.15])
            
        # Scenario B: Partial Match (Good location and size, but slightly over budget)
        elif region_match and space_match and (house['price'] <= user['max_budget'] * 1.15):
            # 40% chance they stretch their budget and save it anyway
            label = np.random.choice([1, 0], p=[0.40, 0.60])
            
        # Scenario C: Dealbreaker Failed (Wrong region entirely OR massively over budget)
        else:
            # 2% tiny chance they click it out of curiosity (noise for the network)
            label = np.random.choice([1, 0], p=[0.02, 0.98])
            
        interactions.append([user['user_id'], house['house_id'], label])

interactions_df = pd.DataFrame(interactions, columns=['user_id', 'house_id', 'watchlist_clicked'])

print(f"Houses Table Shape: {houses_df.shape} (200 properties, 8 columns)")
print(f"Users Table Shape: {users_df.shape} (50 buyers, 4 columns)")
print(f"Interactions Row Count: {len(interactions_df)} total matrix combinations\n")

print("--- SAMPLE USER PROFILE ---")
print(users_df.iloc[0])

print("\n--- SAMPLE HOUSING METRICS ---")
print(houses_df.head(3))

print("\n--- TARGET CLASS DISTRIBUTION ---")
print(interactions_df['watchlist_clicked'].value_counts())
# print(interactions_df.head(1000))

Houses Table Shape: (200, 8) (200 properties, 8 columns)
Users Table Shape: (50, 4) (50 buyers, 4 columns)
Interactions Row Count: 10000 total matrix combinations

--- SAMPLE USER PROFILE ---
user_id                            1
preferred_region    Central Auckland
max_budget                   1620488
min_bedrooms                       1
Name: 0, dtype: object

--- SAMPLE HOUSING METRICS ---
   house_id            region property_type   price      cv  bedrooms  \
0      1000  Central Auckland     Townhouse  974445  919159         1   
1      1001    South Auckland         House  507338  509190         1   
2      1002     West Auckland         House  664884  677819         1   

   school_rating   outside_space  
0              8       Deck Only  
1              7  Large Backyard  
2             10  Large Backyard  

--- TARGET CLASS DISTRIBUTION ---
watchlist_clicked
0    8802
1    1198
Name: count, dtype: int64


In [14]:
# Convert text categories into explicit 1s and 0s
encoded_houses = pd.get_dummies(houses_df, columns=['region', 'property_type', 'outside_space']).astype(int)
encoded_users = pd.get_dummies(users_df, columns=['preferred_region']).astype(int)

print(encoded_houses.iloc[0])
print()
print(encoded_users.iloc[0])

house_id                           1000
price                            974445
cv                               919159
bedrooms                              1
school_rating                         8
region_Central Auckland               1
region_North Shore                    0
region_South Auckland                 0
region_West Auckland                  0
property_type_Apartment               0
property_type_House                   0
property_type_Townhouse               1
outside_space_Balcony                 0
outside_space_Deck Only               1
outside_space_Large Backyard          0
outside_space_None                    0
outside_space_Small Courtyard         0
Name: 0, dtype: int64

user_id                                    1
max_budget                           1620488
min_bedrooms                               1
preferred_region_Central Auckland          1
preferred_region_North Shore               0
preferred_region_South Auckland            0
preferred_region_West Auckl

In [17]:
#we need to tone down to (0-1) the price cv bedrooms and school ratings in houses and min bedrooms and max budget in users
scaled_houses = encoded_houses.copy()
scaled_users = encoded_users.copy()

#the cols that need to be toned down
house_cols = ['price', 'cv', 'bedrooms', 'school_rating']
user_cols = ['max_budget', 'min_bedrooms']

for col in house_cols:
    col_min = scaled_houses[col].min()
    col_max = scaled_houses[col].max()

    scaled_houses[col] = (scaled_houses[col] - col_min) / (col_max - col_min)

for col in user_cols:
    col_min = scaled_users[col].min()
    col_max = scaled_users[col].max()

    scaled_users[col] = (scaled_users[col] - col_min) / (col_max - col_min)

In [18]:
print("--- PREPROCESSED HOUSE ROW (FIRST ENTRY) ---")
print(scaled_houses.drop(columns=['house_id']).iloc[0])

--- PREPROCESSED HOUSE ROW (FIRST ENTRY) ---
price                            0.339820
cv                               0.304036
bedrooms                         0.000000
school_rating                    0.777778
region_Central Auckland          1.000000
region_North Shore               0.000000
region_South Auckland            0.000000
region_West Auckland             0.000000
property_type_Apartment          0.000000
property_type_House              0.000000
property_type_Townhouse          1.000000
outside_space_Balcony            0.000000
outside_space_Deck Only          1.000000
outside_space_Large Backyard     0.000000
outside_space_None               0.000000
outside_space_Small Courtyard    0.000000
Name: 0, dtype: float64


In [27]:
#drop the id's from our scaled tables and create a copy
user_features_only = scaled_users.drop(columns = ['user_id'])
house_features_only = scaled_houses.drop(columns = ['house_id'])

X_user_list = []
X_house_list = []
Y_list = []
for _, row in interactions_df.iterrows():
    uid = row['user_id']
    hid = row['house_id']
    label = row['watchlist_clicked']

    user_vector = user_features_only[scaled_users['user_id'] == uid].values[0]
    house_vector = house_features_only[scaled_houses['house_id'] == hid].values[0]
    
    X_user_list.append(user_vector)
    X_house_list.append(house_vector)
    Y_list.append(label)

#convert them into an array
X_user = np.array(X_user_list).T
X_house = np.array(X_house_list).T
Y = np.array(Y_list).reshape(1, -1)

In [28]:
print("--- FINAL NEURAL NETWORK INPUT SHAPES ---")
print(f"X_user shape : {X_user.shape}  -> (Features, Total Pairs)")
print(f"X_house shape: {X_house.shape}  -> (Features, Total Pairs)")
print(f"Y shape      : {Y.shape}  -> (1, Total Pairs)")

--- FINAL NEURAL NETWORK INPUT SHAPES ---
X_user shape : (6, 10000)  -> (Features, Total Pairs)
X_house shape: (16, 10000)  -> (Features, Total Pairs)
Y shape      : (1, 10000)  -> (1, Total Pairs)
